# 05 — Per-chunk Context Attribution (debug)

Debug harness for `dare.attribution`, run the **per-chunk** way:
ContextCite is applied **once per retrieved chunk**, each chunk small enough that
every `prompt_logprobs` call stays under the worker's ~2k ceiling, throttled +
retried so a burst can't crash it.

**What this notebook does and does not do:**

- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock (a real captured backend payload), so `fetch_backend`
  is never invoked.
- **Still calls the Zingplay vLLM endpoint** for ContextCite (`prompt_logprobs=1`),
  one chunk at a time.

Flow: `load mock → response → per-chunk ContextCite`. (The earlier whole-context
approach — scoring all chunks in one run — was removed: it exceeds the worker's
ceiling. See git history if you need it.)

## Parameters

All knobs for the run live here. Tweak and re-run top-to-bottom.

In [1]:
# ============================ Parameters ============================

# Mock backend payload to debug against (no live backend call is made).
MOCK_PATH = 'chat_output.txt'
QUERY     = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Response cap (chars). The response rides in every prompt_logprobs call, so a
# shorter response leaves more of the per-chunk budget for context.
MAX_RESPONSE_CHARS = 500

# ContextCite ablation settings.
NUM_ABLATIONS      = 32     # number of masked context variants scored per chunk
ABLATION_KEEP_PROB = 0.5    # probability each source is kept in a given ablation
BATCH_SIZE         = 1

# --- Per-chunk pipeline ---
# Run ContextCite once PER CHUNK, each chunk capped so the full-context call stays
# under the worker's ~2k ceiling. Heading + table title-row are fixed scaffolding;
# paragraphs and table data-rows are the ablatable sources.
PER_CHUNK_MAX_CHARS = 2000   # trim a chunk's context above this (keeps calls <= ceiling)
TOP_K               = 3      # how many top sources to print per chunk

# --- Graceful calls: throttle + retry (applied to the module) ---
# The mini worker falls over under bursts of prompt_logprobs calls. Space calls out
# and ride through transient 5xx/connection errors. Operational only — changes no score.
API_CALL_DELAY    = 1.0   # seconds to wait BEFORE each call
API_MAX_RETRIES   = 4     # total attempts per call
API_RETRY_BACKOFF = 3.0   # linear backoff: wait = backoff * attempt_number

# Logging: when True the module logs per-call token counts (and the size/HTTP
# status of any failing call) at INFO.
VERBOSE_API_LOGGING = False

## Setup — import the module and inspect its config

In [2]:
import sys, json, logging
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs.
import dare.attribution as cc_module
from dare.attribution import prepare_inputs, make_tokenizer, APIModel
from context_cite import ContextCiter

# Response cap from Parameters. Per-chunk uses raw chunks, so we don't truncate the
# (unused) concatenated context — only the response matters for call size.
cc_module.MINI_MAX_RESPONSE_CHARS = MAX_RESPONSE_CHARS
cc_module.MINI_MAX_CONTEXT_CHARS  = None

# Apply graceful-call settings (throttle + retry) to the module's call chokepoint.
cc_module.API_CALL_DELAY    = API_CALL_DELAY
cc_module.API_MAX_RETRIES   = API_MAX_RETRIES
cc_module.API_RETRY_BACKOFF = API_RETRY_BACKOFF

# Logging: keep the root quiet (httpx/transformers stay at WARNING) but turn the
# module's per-call token logging up to INFO so it surfaces in the notebook.
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s', force=True)
logging.getLogger('dare.attribution').setLevel(
    logging.INFO if VERBOSE_API_LOGGING else logging.WARNING
)

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Response cap (chars):', cc_module.MINI_MAX_RESPONSE_CHARS)
print(f'Throttle / retry    : delay={API_CALL_DELAY}s, retries={API_MAX_RETRIES}, backoff={API_RETRY_BACKOFF}s')
print('Verbose API logging :', VERBOSE_API_LOGGING)

# Sanity: we need the Zingplay endpoint for ContextCite, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — ContextCite needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

Module under test   : /Users/lap15114/projects/dairy_vng/dare/attribution.py
LLM endpoint (ZP)   : https://chat.zingplay.com/api/v1/chat/completions
Model               : local-model-mini
LLM API key set     : True
Response cap (chars): 500
Throttle / retry    : delay=1.0s, retries=4, backoff=3.0s
Verbose API logging : False

Ready: Zingplay endpoint configured, backend will NOT be called.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lap15114/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Health check — is the model backend alive?

A trivial `"hello"` call (`max_tokens=1`) against the Zingplay endpoint, run
*before* any heavy calls. If this 500s, the `local-model-mini` worker is down /
unreachable (e.g. crashed by an earlier oversized prompt) — fix the backend
before going further; no amount of context trimming will help. This mirrors
`dare/tests/test_zp_api.py`.

In [3]:
import requests

r = requests.post(
    cc_module.LLM_URL,
    headers={'Authorization': f'Bearer {cc_module.API_KEY}'},
    json={
        'model': cc_module.MODEL,
        'messages': [{'role': 'user', 'content': 'hello'}],
        'max_tokens': 1,
        'temperature': 0,
    },
    timeout=30,
)

print(f'POST {cc_module.LLM_URL}')
print(f'model       : {cc_module.MODEL}')
print(f'status      : {r.status_code}')
print(f'body (head) : {r.text[:200]}')

if r.ok:
    print('\nBackend is ALIVE — safe to proceed.')
else:
    print('\n*** Backend is DOWN / unreachable. ***')
    print('A trivial "hello" call failed, so this is a backend problem, not a'
          ' context-size problem. Restart / verify the model worker before running'
          ' the rest of the notebook.')

assert r.ok, f'Model backend not healthy (HTTP {r.status_code}) — fix the endpoint before continuing.'

POST https://chat.zingplay.com/api/v1/chat/completions
model       : local-model-mini
status      : 200
body (head) : {"id":"chatcmpl-a6970a0735252796","created":1781679713,"model":"local-model-mini","object":"chat.completion","choices":[{"finish_reason":"length","index":0,"message":{"role":"assistant","reasoning_con

Backend is ALIVE — safe to proceed.


## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [4]:
# MOCK_PATH and QUERY come from the Parameters cell.
# Load directly from file — fetch_backend() is deliberately NOT used here.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', Path(MOCK_PATH).resolve())
print('Query               :', QUERY)
print('Top-level keys      :', list(data.keys()))
print()
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print(f'mode                : {data.get("mode")}')
print(f'usage               : {data.get("usage")}')
print()
print('--- raw answer ---')
print(data['answer'])
print()
print('--- knowledge source chunks ---')
for i, src in enumerate(data['knowledge_sources']):
    snip = ' '.join(src.get('content', '').split())
    print(f'  [{i}] score={src.get("score", 0):.3f}  {src.get("document_id", "<no doc>")}')
    print(f'       {snip}...')

Loaded mock payload : /Users/lap15114/projects/dairy_vng/notebooks/chat_output.txt
Query               : Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?
Top-level keys      : ['answer', 'knowledge_sources', 'memory_sources', 'reasoning_trace', 'tools_used', 'session_id', 'usage', 'mode', 'ablation']

answer              : 639 chars
knowledge_sources   : 10 chunks
mode                : agent
usage               : {'llm_calls': 10, 'input_tokens': 31190, 'output_tokens': 2912, 'total_tokens': 34102}

--- raw answer ---
Theo tài liệu chính sách nội bộ VNG, số ngày phép năm được quy định như sau:

| Cấp bậc | Số ngày phép năm |
|---------|:----------------:|
| Cấp bậc 2, 3, 4, 5 | **16 ngày** |
| Cấp bậc 1 | **14 ngày** |

Ngoài ra, nhân viên làm việc tại VNG trên 03 năm sẽ được **cộng thêm 01 ngày Phép thâm niên** cứ mỗi 03 năm.

**Một số lưu ý:**
- Phép năm được tính từ ngày 01/01 đến 31/12 hằng năm.
- Phép năm chưa sử dụng hết được bảo lưu sang năm tiếp theo (tối đa 5 ngày) và phải

## Stage 2 — Response to attribute

`prepare_inputs` applies the response cap and returns the answer we'll attribute.
We only need the **response** here — per-chunk attribution scores it against each
raw chunk from `data`, so the concatenated context it also returns is unused.

In [5]:
# We only need the (capacity-capped) RESPONSE; the concatenated context is unused
# because per-chunk attribution works from the raw chunks in `data`.
_context, response = prepare_inputs(data)

print(f'Response ({len(response)} chars):')
print(response)
print(f'\nChunks available for per-chunk attribution: {len(data["knowledge_sources"])}')

Chunks: 10/10 | Context: 7712 chars | Response: 388 chars
Response (388 chars):
Theo tài liệu chính sách nội bộ VNG, số ngày phép năm được quy định như sau:

| Cấp bậc | Số ngày phép năm |
|---------|:----------------:|
| Cấp bậc 2, 3, 4, 5 | **16 ngày** |
| Cấp bậc 1 | **14 ngày** |

Ngoài ra, nhân viên làm việc tại VNG trên 03 năm sẽ được **cộng thêm 01 ngày Phép thâm niên** cứ mỗi 03 năm.

**Một số lưu ý:**
- Phép năm được tính từ ngày 01/01 đến 31/12 hằng năm.

Chunks available for per-chunk attribution: 10


## Stage 3 — Per-chunk ContextCite with the paragraph/table-row partitioner

Run ContextCite **once per chunk**, each chunk capped at `PER_CHUNK_MAX_CHARS` so
the full-context call stays under the worker's ~2k ceiling, throttled + retried so
a burst can't crash the worker.

**Partitioner (`ParagraphTableContextPartitioner`):**
- **Heading** and a **table's title/separator row** are *fixed scaffolding* —
  always present in every ablation, rendered once, never ablated. This frames each
  piece in its proper context without duplication or credit-leakage.
- **Ablatable sources** are the text **paragraphs** (split on `\n\n`) and the
  table **data rows** (one source each, judged with the column header always shown).

**Caveats to keep in mind** (discussed):
- Per-chunk runs share no common baseline → scores are *not* comparable across
  chunks, and redundant chunks (duplicate tables/boilerplate) get over-credited.
- **Position awareness:** the output prints each source's position index. Watch for
  top scores clustering at low indices (right after the heading) — that would hint
  attribution is tracking *proximity* rather than content.
- Chunks that split to `<2` sources (the short boilerplate) are skipped — nothing
  to ablate.

In [6]:
import re
import numpy as np
from typing import Optional
from context_cite.context_partitioner import BaseContextPartitioner


def _is_table_block(lines):
    """A markdown table block: header row, a `---` separator row, then data rows."""
    return (len(lines) >= 2 and '|' in lines[0]
            and bool(re.match(r'^[\s|:\-]+$', lines[1])) and '-' in lines[1])


def _split_bullets(block):
    """Split a text block into bullet/numbered items. Top-level markers only;
    indented sub-bullets ride along with their parent."""
    lines = block.split('\n')
    items, current = [], []
    marker = re.compile(r'^(?:[-*]|\d+\.)\s')
    for ln in lines:
        if marker.match(ln) and current:
            items.append('\n'.join(current))
            current = [ln]
        else:
            current.append(ln)
    if current:
        items.append('\n'.join(current))
    return items


class ParagraphTableContextPartitioner(BaseContextPartitioner):
    """Per-chunk splitter for ContextCite."""

    def __init__(self, context: str) -> None:
        super().__init__(context)
        self._cache = {}

    def split_context(self) -> None:
        text = self.context
        tokens = re.split(r'(\n[ \t]*\n)', text)
        segs = []
        prev_sep = ''
        for i, tok in enumerate(tokens):
            if i % 2 == 1:
                prev_sep = tok
                continue
            if tok == '':
                continue
            lines = tok.split('\n')
            if _is_table_block(lines):
                segs.append({'text': tok, 'fixed': False, 'sep': prev_sep})
            elif tok.lstrip().startswith('#'):
                segs.append({'text': tok, 'fixed': True, 'sep': prev_sep})
            else:
                items = _split_bullets(tok)
                for j, item in enumerate(items):
                    segs.append({
                        'text': item,
                        'fixed': False,
                        'sep': prev_sep if j == 0 else '\n',
                    })
            prev_sep = ''
        self._cache['segments']   = segs
        self._cache['source_idx'] = [k for k, s in enumerate(segs) if not s['fixed']]

    @property
    def segments(self):
        if self._cache.get('segments') is None:
            self.split_context()
        return self._cache['segments']

    @property
    def _source_idx(self):
        if self._cache.get('source_idx') is None:
            self.split_context()
        return self._cache['source_idx']

    @property
    def num_sources(self) -> int:
        return len(self._source_idx)

    def get_source(self, index: int) -> str:
        return self.segments[self._source_idx[index]]['text']

    def get_context(self, mask: Optional[np.ndarray] = None):
        if mask is None:
            mask = np.ones(self.num_sources, dtype=bool)
        out, first, si = '', True, -1
        for seg in self.segments:
            if not seg['fixed']:
                si += 1
                if not bool(mask[si]):
                    continue
            out += ('' if first else seg['sep']) + seg['text']
            first = False
        return out


# Quick self-check.
_p = ParagraphTableContextPartitioner(data['knowledge_sources'][0]['content'])
assert _p.get_context() == data['knowledge_sources'][0]['content'], 'partitioner must round-trip'
print('Partitioner OK. Source counts per chunk (fixed scaffolding excluded):')
for ci, ch in enumerate(data['knowledge_sources']):
    p = ParagraphTableContextPartitioner(ch['content'][:PER_CHUNK_MAX_CHARS])
    print(f'  chunk {ci}: {len(ch["content"]):>4} chars -> {p.num_sources} ablatable sources')

Partitioner OK. Source counts per chunk (fixed scaffolding excluded):
  chunk 0: 1682 chars -> 9 ablatable sources
  chunk 1: 1868 chars -> 9 ablatable sources
  chunk 2:  157 chars -> 1 ablatable sources
  chunk 3:  157 chars -> 1 ablatable sources
  chunk 4:  160 chars -> 1 ablatable sources
  chunk 5:  162 chars -> 1 ablatable sources
  chunk 6:  159 chars -> 1 ablatable sources
  chunk 7: 1450 chars -> 7 ablatable sources
  chunk 8: 1034 chars -> 6 ablatable sources
  chunk 9:  865 chars -> 5 ablatable sources


In [7]:
import pandas as pd
from IPython.display import display

# Per-chunk ContextCite. One faithful run per chunk; every call <= PER_CHUNK_MAX_CHARS,
# throttled + retried via the module settings applied in Setup.
per_chunk_results = {}

for ci, ch in enumerate(data['knowledge_sources']):
    content = ch['content'][:PER_CHUNK_MAX_CHARS]
    part = ParagraphTableContextPartitioner(content)
    heading = next((s['text'] for s in part.segments if s['fixed']), '<no heading>')

    print('=' * 78)
    print(f'CHUNK {ci}  | score={ch.get("score", 0):.3f} | {ch.get("document_id", "")[:60]}')
    print(f'  heading : {heading[:70]}')
    print(f'  sources : {part.num_sources} ablatable')

    if part.num_sources < 2:
        print('  -> <2 sources, nothing to ablate. Skipped.')
        per_chunk_results[ci] = None
        continue

    cc = ContextCiter(
        model=APIModel(response=response),
        tokenizer=make_tokenizer(),
        context=content,
        query=QUERY,
        num_ablations=NUM_ABLATIONS,
        ablation_keep_prob=ABLATION_KEEP_PROB,
        batch_size=BATCH_SIZE,
        partitioner=part,
    )
    try:
        cc.solver.lasso_alpha = 0.001
        attrs = cc.get_attributions(as_dataframe=False, verbose=False)
    except Exception as e:
        print(f'  -> FAILED: {e.__class__.__name__}: {str(e)[:160]}')
        per_chunk_results[ci] = None
        continue

    per_chunk_results[ci] = (part, attrs)
    order = np.argsort(attrs)[::-1]
    order = np.argsort(attrs)[::-1]
    df = pd.DataFrame({
        'pos':    [int(i) for i in order],
        'score':  [attrs[int(i)] for i in order],
        'source': [part.get_source(int(i)) for i in order],
    })
    styled = (df.style
                .hide(axis='index')
                .background_gradient(cmap='Greens', subset=['score'])
                .format({'score': '{:+.4f}'})
                .set_properties(subset=['source'],
                                **{'text-align': 'left', 'white-space': 'pre-wrap'}))
    display(styled)


print('\nDone. per_chunk_results[chunk_index] = (partitioner, attribution_array) or None.')

CHUNK 0  | score=0.870 | HR/Thoi gian lam viec va thoi gio nghi ngoi/CS-HR.CB-002-11-
  heading : ### 3.1.2. Quy định cách sử dụng
  sources : 9 ablatable


WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1557 > 1024). Running this sequence through the model will result in indexing errors


  0%|          | 0/32 [00:00<?, ?it/s]

/Users/lap15114/projects/dairy_vng/venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/Users/lap15114/projects/dairy_vng/venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1242 > 1024). Running this sequence through the model will result in indexing errors


pos,score,source
7,+15.6507,"- Nhân viên cấp bậc 5 thuộc nhóm có thời gian làm việc linh động, không áp dụng việc chuyển Phép năm chưa sử dụng hết thành tiền."
6,+6.3173,- Ngày phép lẻ được làm tròn theo nguyên tắc sau: - Từ 0.5 ngày trở xuống: làm tròn thành 0.5 ngày - Trên 0.5 ngày: làm tròn thành 01 ngày
8,+4.2548,- Việc sử dụng Phép năm bù vào thời gian bảo trước (nếu có) phải có sự phê duyệt của DH trước thời gian thực nghỉ.
1,+3.5081,- Quy định ứng trước phép không áp dụng cho Nhân viên đã thông báo nghỉ việc và trong thời gian chờ nghỉ việc.
3,+1.8606,- Các trường hợp nghỉ hưởng lương dài ngày theo Thỏa thuận tạm hoãn HĐLĐ sẽ không được hưởng Phép năm kể từ tháng nghỉ thứ 02 trở đi cho đến khi đi làm trở lại.
4,+1.3547,"- Trường hợp Nhân viên nghỉ việc, số ngày còn lại của Phép năm hiện tại được hưởng tính đến thời điểm nghỉ việc nếu chưa sử dụng hết sẽ được tất toán trong Quyết toán thôi việc."
5,-3.4753,| | Tổng lương cơ bản và mức thưởng V_Bonus | | :---------------- | :---------------------------------- | | Tất toán phép năm | tối đa của tháng trên HĐLĐ |
0,-8.4595,- Số ngày nghỉ tối đa Nhân viên có thể đăng ký nghỉ = Số ngày còn lại của Phép năm thực tế tính đến thời điểm đăng ký + tối đa 05 ngày tạm ứng Phép năm của năm (trong còn lại tính đến ngày 31/12 hoặc ngày hết hiệu lực của HĐLĐ hiện tại nếu trước ngày nào đến trước).
2,-16.0566,"- Công ty khuyến khích Nhân viên có kế hoạch sử dụng phép năm hợp lý trong từng năm. Số phép năm chưa sử dụng hết được bảo lưu sang tháng hai (theo dõi tối đa 2 ngày) và phải được sử dụng không muộn hơn 31/03 năm đó. Vượt quá thời điểm này, số ngày phép bảo lưu không sử dụng hết mặc nhiên sẽ bị hủy bỏ. Các ngày phép bảo lưu không được chuyển đổi thành tiền ở bất cứ trường hợp nào."


...

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1162 > 1024). Running this sequence through the model will result in indexing errors


  0%|          | 0/32 [00:00<?, ?it/s]

/Users/lap15114/projects/dairy_vng/venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/Users/lap15114/projects/dairy_vng/venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


pos,score,source
4,+117.1203,- Nhân viên làm việc tại Công ty trên 03 năm thì cứ mỗi 03 năm sẽ được cộng thêm 01 ngày Phép thâm niên.
0,+25.7731,"- Phép năm là phép nghỉ hưởng niên hưởng nguyên lương, được tính từ ngày 01/01 đến ngày 31/12 của năm, cụ thể như sau:"
3,-0.6058,"- Nhân viên thay đổi cấp bậc nếu có thay đổi định mức, Phép năm theo cấp bậc mới được áp dụng theo nguyên tắc: - Ngày hiệu lực thay đổi cấp bậc từ ngày 01 đến ngày 20: định mức Phép năm theo cấp bậc mới được áp dụng từ tháng hiệu lực thay đổi cấp bậc. - Ngày hiệu lực thay đổi cấp bậc từ ngày 21 đến cuối tháng: định mức Phép năm theo cấp bậc mới được áp dụng từ tháng kế tiếp tháng hiệu lực thay đổi cấp bậc."
1,-10.0528,"| Cấp bậc | Quy phép năm | | :------ | :----------- | | Cấp bậc 2, 3, 4, 5 | 16 ngày | | Cấp bậc 1 | 14 ngày |"
2,-41.2565,- Quy phép năm Thực tế được tính theo tỷ lệ số tháng làm việc thực tế trong năm.



Done. per_chunk_results[chunk_index] = (partitioner, attribution_array) or None.


In [8]:
import numpy as np

# --- Delta test on the last chunk's table source ---
CHUNK_IDX = max(k for k, v in per_chunk_results.items() if v is not None)
part, attrs = per_chunk_results[CHUNK_IDX]

cand = [i for i in range(part.num_sources) if part.get_source(i).lstrip().startswith('|')]
SOURCE_IDX = cand[0] if cand else int(np.argmax(attrs))

print(f'Chunk {CHUNK_IDX}, source pos[{SOURCE_IDX}] (Lasso score={attrs[SOURCE_IDX]:+.4f})')
print(part.get_source(SOURCE_IDX))
print('-' * 60)

# Rebuild the same cc used in the loop.
content = data['knowledge_sources'][CHUNK_IDX]['content'][:PER_CHUNK_MAX_CHARS]
cc = ContextCiter(
    model=APIModel(response=response),
    tokenizer=make_tokenizer(),
    context=content,
    query=QUERY,
    num_ablations=NUM_ABLATIONS,
    ablation_keep_prob=ABLATION_KEEP_PROB,
    batch_size=BATCH_SIZE,
    partitioner=part,
)

# Build user_content from the instance's own template (same as attribution path).
def logprob_for_mask(mask):
    ctx = part.get_context(mask)
    user_content = cc.prompt_template.format(context=ctx, query=QUERY)
    toks = cc_module._api_response_token_logprobs(user_content, response)
    return sum(lp for _, lp in toks)

full_mask = np.ones(part.num_sources, dtype=bool)
drop_mask = full_mask.copy()
drop_mask[SOURCE_IDX] = False

lp_full = logprob_for_mask(full_mask)
lp_drop = logprob_for_mask(drop_mask)
delta   = lp_full - lp_drop

print(f'logprob, full context   : {lp_full:.3f}')
print(f'logprob, source removed  : {lp_drop:.3f}')
print(f'delta (full - ablated)   : {delta:+.3f}')

Chunk 9, source pos[1] (Lasso score=-10.0528)
| Cấp bậc | Quy phép năm |
| :------ | :----------- |
| Cấp bậc 2, 3, 4, 5 | 16 ngày |
| Cấp bậc 1 | 14 ngày |
------------------------------------------------------------
logprob, full context   : -107.901
logprob, source removed  : -133.326
delta (full - ablated)   : +25.425
